In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold,train_test_split
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,confusion_matrix,roc_auc_score

data=load_breast_cancer()
x,y=data.data,data.target
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

base_models=[('Logistic',make_pipeline(StandardScaler(),LogisticRegression(max_iter=100))),
             ('random_forest',make_pipeline(StandardScaler(),RandomForestClassifier(n_estimators=100,random_state=42))),
             ('gradient_boosting',make_pipeline(StandardScaler(),GradientBoostingClassifier(n_estimators=100,learning_rate=0.01,max_depth=2,random_state=42)))]

stacking_model=StackingClassifier(estimators=base_models,final_estimator=LogisticRegression(max_iter=100),cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42),stack_method='predict_proba')

stacking_model.fit(x_train,y_train)
y_pred=stacking_model.predict(x_test)
y_prob=stacking_model.predict_proba(x_test)[:,1]

print("Stacking Accuracy:", round(
    accuracy_score(y_test, y_pred), 4
))
print("Stacking ROC-AUC:", round(
    roc_auc_score(y_test, y_prob), 4
))


Stacking Accuracy: 0.9737
Stacking ROC-AUC: 0.9964
